# 第9章 サポートベクターマシンと凸最適化 ― デモノートブック

この章の主張を数値で確かめる。マージン最大化が凸二次計画（式 9.3）になること、
その双対（式 9.12、式 9.18）を自分で解くと `sklearn.svm.SVC` と一致すること、
KKT の相補性条件（式 9.14）から支持ベクトルの疎性が出ること、
ソフトマージンがヒンジ損失＋$\ell_2$ 正則化（式 9.23）に等しいこと、
カーネル化（式 9.24、式 9.25）で三日月データが分離できること、
そして一つ抜き交差確認の誤り率が支持ベクトル割合で抑えられること
（定理「LOO 誤差の支持ベクトル数による評価」、9.10 節）を順に見る。

## 目次

- 9.1 ハードマージン SVM ― マージンと分離超平面（9.3 節、図 9.1）
- 9.2 主問題と双対問題 ― 双対を自分で解く（9.5 節、式 9.13）
- 9.3 KKT 条件と支持ベクトルの疎性（式 9.14、式 9.15）
- 9.4 ソフトマージンと $C$ の効果（9.6 節、図 9.2）
- 9.5 ヒンジ損失と他の損失関数（9.7 節、図 9.3）
- 9.6 カーネル SVM と $(\gamma, C)$ グリッド（9.8 節、図 9.4）
- 9.7 LOO 誤差 $\le |\mathcal{S}|/n$ の検証（9.10 節）
- 演習 / 演習の解答

行列の規約：**データ行列は列がサンプル**（$\boldsymbol{X}\in\mathbb{R}^{d\times n}$）である。
scikit-learn は行がサンプルなので、渡すときに `X.T` と転置する。

注意：準備セルが定義する `C` は色の辞書である。SVM の正則化パラメータは
本ノートブックでは `Creg` と書いて衝突を避ける。

## 準備

最初にこのセルを実行する。日本語フォントの設定（Colab には既定で入っていない）と、
以降で使うライブラリの読み込みを行う。フォントの導入に失敗した場合は
図のラベルが自動的に英語に切り替わる（`L()` 関数）。

In [ ]:
import subprocess, sys, warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings("ignore", category=UserWarning)


def _setup_japanese_font():
    """日本語が出せるフォントを探し、なければ入れる。成功したら True。"""
    cands = ["IPAexGothic", "IPAGothic", "Noto Sans CJK JP", "Noto Sans JP",
             "TakaoGothic", "Yu Gothic", "Hiragino Sans"]
    have = {f.name for f in fm.fontManager.ttflist}
    for name in cands:
        if name in have:
            matplotlib.rcParams["font.family"] = name
            return True
    # Colab 想定：pip で導入する
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "japanize-matplotlib"], check=True, timeout=180)
        import japanize_matplotlib  # noqa: F401  読み込むだけで設定される
        return True
    except Exception:
        pass
    # 予備：apt で IPA フォント
    try:
        subprocess.run("apt-get -qq -y install fonts-ipafont-gothic",
                       shell=True, check=True, timeout=300)
        fm._load_fontmanager(try_read_cache=False)
        matplotlib.rcParams["font.family"] = "IPAGothic"
        return True
    except Exception:
        return False


JP = _setup_japanese_font()


def L(ja, en):
    """日本語フォントが使えれば ja、駄目なら en を返す（図のラベル用）。"""
    return ja if JP else en


matplotlib.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "figure.dpi": 110, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.unicode_minus": False,
})

# 講義ノートの図と同じ色
C = {"blue": "#1f4e79", "red": "#c0392b", "green": "#1e8449",
     "orange": "#d68910", "purple": "#6a4c93", "gray": "#7f8c8d"}

print("日本語フォント:", "有効" if JP else "無効（図のラベルは英語になる）")
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

## 9.1 ハードマージン SVM ― マージンと分離超平面

線形分離可能なデータでは分離超平面は無数にあるが、正準形（式 9.2）のもとで
$\frac12\|\boldsymbol{w}\|^2$ を最小化すると解は一意に定まる（命題「凸二次計画であること・解の一意性」）。
このとき幾何マージンは $1/\|\boldsymbol{w}\|$、マージン領域の幅は $2/\|\boldsymbol{w}\|$ である
（命題「マージン領域の幅」）。図 9.1 に対応する図を描き、
マージン境界に乗る点だけが支持ベクトルになることを見る。

あわせて講義ノートの例「3 点の例を手で解く」
（$\boldsymbol{x}_1=(0,0)^\top, y_1=-1$、$\boldsymbol{x}_2=(2,0)^\top$、$\boldsymbol{x}_3=(0,2)^\top$、$y=+1$）
の手計算 $\boldsymbol{w}^\star=(1,1)^\top$、$b^\star=-1$、$\gamma=1/\sqrt2$ を数値で確認する。

In [ ]:
from sklearn.datasets import make_blobs, make_moons
from sklearn.svm import SVC

# 線形分離可能な 2 クラス。X は d×n（列がサンプル）
Xb, yb01 = make_blobs(n_samples=40, centers=[[-2.0, -2.0], [2.0, 2.0]],
                      cluster_std=0.8, random_state=0)
X, y = Xb.T, 2.0 * yb01 - 1.0            # ラベルを {-1,+1} に

# C を十分大きくとればハードマージン。sklearn は n×d 規約なので転置して渡す
hard = SVC(kernel="linear", C=1e6, tol=1e-12).fit(X.T, y)
w, b = hard.coef_[0], float(hard.intercept_[0])
gamma_geo = 1.0 / np.linalg.norm(w)

print("w =", np.round(w, 4), "  b =", round(b, 4))
print("幾何マージン 1/||w|| =", round(gamma_geo, 4),
      "  帯の幅 2/||w|| =", round(2 * gamma_geo, 4))
print("支持ベクトル数 |S| =", hard.support_.size, "/ n =", X.shape[1])
print("min_i y_i f(x_i) =", round(float((y * hard.decision_function(X.T)).min()), 8),
      "（正準形なら 1）")

# 例「3 点の例を手で解く」の確認
X3 = np.array([[0.0, 2.0, 0.0],
               [0.0, 0.0, 2.0]])          # d×n = 2×3
y3 = np.array([-1.0, 1.0, 1.0])
c3 = SVC(kernel="linear", C=1e6, tol=1e-12).fit(X3.T, y3)
print("3 点の例: w* =", np.round(c3.coef_[0], 6),
      " b* =", round(float(c3.intercept_[0]), 6),
      " 1/||w*|| =", round(1 / np.linalg.norm(c3.coef_[0]), 4),
      " 支持ベクトル数 =", c3.support_.size)

In [ ]:
def plot_linear_svm(ax, X, y, clf, title):
    """d×n のデータと線形 SVM の境界・マージン境界・支持ベクターを描く。"""
    ax.scatter(*X[:, y > 0], s=26, color=C["red"], label=L("クラス +1", "class +1"))
    ax.scatter(*X[:, y < 0], s=26, color=C["blue"], label=L("クラス -1", "class -1"))
    sv = clf.support_
    ax.scatter(*X[:, sv], s=140, facecolors="none", edgecolors=C["green"],
               linewidths=1.6, label=L("支持ベクトル", "support vectors"))
    x1 = np.linspace(X[0].min() - 1, X[0].max() + 1, 200)
    x2 = np.linspace(X[1].min() - 1, X[1].max() + 1, 200)
    G1, G2 = np.meshgrid(x1, x2)
    Z = clf.decision_function(np.c_[G1.ravel(), G2.ravel()]).reshape(G1.shape)
    ax.contour(G1, G2, Z, levels=[-1, 0, 1], colors=[C["gray"], "k", C["gray"]],
               linestyles=["--", "-", "--"], linewidths=[1.0, 1.6, 1.0])
    ax.set_title(title)
    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")


fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
plot_linear_svm(axes[0], X, y, hard,
                L("ハードマージン SVM（幅 %.3f）" % (2 * gamma_geo),
                  "hard-margin SVM (width %.3f)" % (2 * gamma_geo)))
axes[0].legend(fontsize=9, loc="upper left")

# マージンを最大化しない分離超平面との比較（同じデータを分離はできる）
axes[1].scatter(*X[:, y > 0], s=26, color=C["red"])
axes[1].scatter(*X[:, y < 0], s=26, color=C["blue"])
gx = np.linspace(X[0].min() - 1, X[0].max() + 1, 100)
for slope, off, col, lab in [(-w[0] / w[1], -b / w[1], "k",
                              L("最大マージン解", "max-margin")),
                             (-0.4, 0.9, C["orange"], L("別の分離超平面 1", "another 1")),
                             (-2.2, -0.6, C["purple"], L("別の分離超平面 2", "another 2"))]:
    axes[1].plot(gx, slope * gx + off, color=col, lw=1.6, label=lab)
axes[1].set_ylim(X[1].min() - 1, X[1].max() + 1)
axes[1].set_title(L("分離超平面は無数にある", "separating hyperplanes are not unique"))
axes[1].set_xlabel("$x_1$"); axes[1].set_ylabel("$x_2$")
axes[1].legend(fontsize=9, loc="upper left")
plt.show()

マージン境界（破線）に乗っている点だけが支持ベクトル（緑の丸）である。
右図のように分離するだけならほかにいくらでも直線が引けるが、
もっとも近い点までの距離を最大にするものは一つしかない。

3 点の例では手計算どおり $\boldsymbol{w}^\star=(1,1)^\top$、$b^\star=-1$、
$1/\|\boldsymbol{w}^\star\|=0.7071$、3 点すべてが支持ベクトルという結果が返る。

## 9.2 主問題と双対問題 ― 双対を自分で解く

双対問題（式 9.12、行列形は式 9.13）

$$\max_{\boldsymbol\alpha}\ \boldsymbol{1}^\top\boldsymbol\alpha-\tfrac12\boldsymbol\alpha^\top \boldsymbol{Q}\boldsymbol\alpha,
\qquad \boldsymbol{Q}=(\boldsymbol{y}\boldsymbol{y}^\top)\odot\boldsymbol{K},\qquad
0\le\alpha_i\le C,\ \boldsymbol{y}^\top\boldsymbol\alpha=0$$

は $n$ 変数の凸二次計画である。符号を反転して最小化の形にし、勾配 $\boldsymbol{Q}\boldsymbol\alpha-\boldsymbol{1}$ を
解析的に与えて `scipy.optimize.minimize` の SLSQP に渡す（講義ノートのリスト 9.1）。
主解は $\boldsymbol{w}=\sum_i\alpha_iy_i\boldsymbol{x}_i$（式 9.10）で復元し、
$b$ は $0<\alpha_i<C$ の点（自由な支持ベクトル）だけを使って推定する（9.6 節の警告）。

In [ ]:
from scipy.optimize import minimize


def svm_dual(K, y, Creg):
    """max_a 1^T a - (1/2) a^T Q a  s.t. 0 <= a <= Creg, y^T a = 0 を SLSQP で解く。"""
    Q = (y[:, None] * y[None, :]) * K              # Q_ij = y_i y_j K_ij
    fun = lambda a: 0.5 * a @ Q @ a - a.sum()      # 符号を反転して最小化
    jac = lambda a: Q @ a - np.ones(len(y))        # 勾配 Q a - 1
    cons = [{"type": "eq", "fun": lambda a: a @ y, "jac": lambda a: y}]
    return minimize(fun, np.zeros(len(y)), jac=jac, bounds=[(0, Creg)] * len(y),
                    constraints=cons, method="SLSQP",
                    options={"maxiter": 500, "ftol": 1e-12}).x


rng = np.random.default_rng(0)
Xd = np.vstack([rng.normal([-1.5, -1.0], 0.8, (20, 2)),
                rng.normal([1.5, 1.0], 0.8, (20, 2))]).T      # d×n : 列がサンプル
yd = np.r_[-np.ones(20), np.ones(20)]
Creg = 1.0

alpha = svm_dual(Xd.T @ Xd, yd, Creg)          # 線形カーネル K = X^T X（n×n）
w_d = Xd @ (alpha * yd)                        # w = sum_i a_i y_i x_i（d 次元）
free = (alpha > 1e-6) & (alpha < Creg - 1e-6)  # 0 < a_i < C の点だけで b を推定
b_d = float(np.mean(yd[free] - Xd[:, free].T @ w_d))

ref = SVC(kernel="linear", C=Creg, tol=1e-12).fit(Xd.T, yd)   # sklearn は n×d 規約
print("自作の双対  w =", np.round(w_d, 6), " b =", round(b_d, 8))
print("sklearn     w =", np.round(ref.coef_[0], 6), " b =", round(float(ref.intercept_[0]), 8))
print("係数の最大差   =", f"{np.abs(w_d - ref.coef_[0]).max():.2e}")
print("判別値の最大差 =",
      f"{np.abs((Xd.T @ w_d + b_d) - ref.decision_function(Xd.T)).max():.2e}")
print("双対の最適値 W(a*) =", round(float(alpha.sum() - 0.5 * alpha @ ((yd[:, None] * yd[None, :]) * (Xd.T @ Xd)) @ alpha), 6),
      " / 主の目的値 (1/2)||w||^2 + C sum xi =",
      round(float(0.5 * w_d @ w_d + Creg * np.maximum(0, 1 - yd * (Xd.T @ w_d + b_d)).sum()), 6))

自作の双対解と `SVC` は係数・判別値ともに上の出力の桁で一致する。
強双対性（定理「ソフトマージン SVM の双対問題」の直前の議論）から
双対の最適値 $W(\boldsymbol\alpha^\star)$ とヒンジ形の主目的値
$\frac12\|\boldsymbol{w}\|^2+C\sum_i\xi_i$（式 9.22）も一致する。

## 9.3 KKT 条件と支持ベクトルの疎性

相補性条件（式 9.14）$\alpha_i^\star\bigl(1-y_if(\boldsymbol{x}_i)\bigr)=0$ から

- $\alpha_i=0$ ならマージンの外側（$y_if(\boldsymbol{x}_i)\ge1$）、
- $0<\alpha_i<C$ ならちょうどマージン境界上（$y_if(\boldsymbol{x}_i)=1$）、
- $\alpha_i=C$ ならマージン侵入または誤分類（$y_if(\boldsymbol{x}_i)\le1$）

の三分類（定理「支持ベクトルの三分類」）が出る。上で解いた $\boldsymbol\alpha$ で確かめる。

In [ ]:
f_val = Xd.T @ w_d + b_d
yf = yd * f_val
zero = alpha <= 1e-6
bnd = alpha >= Creg - 1e-6

print(f"(a) alpha=0     : {zero.sum():2d} 点,  min y_i f(x_i) = {yf[zero].min():.4f} （>= 1 のはず）")
print(f"(b) 0<alpha<C   : {free.sum():2d} 点,  max |y_i f(x_i) - 1| = {np.abs(yf[free] - 1).max():.2e}")
print(f"(c) alpha=C     : {bnd.sum():2d} 点,  max y_i f(x_i) = {yf[bnd].max() if bnd.any() else float('nan'):.4f} （<= 1 のはず）")
print("alpha_i > 1e-6 の個数 =", int((alpha > 1e-6).sum()),
      " / sklearn の n_support_.sum() =", int(ref.n_support_.sum()))
print("sum_i alpha_i y_i =", f"{float(alpha @ yd):.2e}", "（等式制約）")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(np.sort(alpha)[::-1], "o-", ms=4, color=C["blue"])
axes[0].axhline(Creg, color=C["red"], ls="--", lw=1.0, label="$C$")
axes[0].set_xlabel(L("大きい順の番号", "index (sorted)"))
axes[0].set_ylabel(r"$\alpha_i$")
axes[0].set_title(L("双対変数の疎性", "sparsity of dual variables"))
axes[0].legend(fontsize=9)

axes[1].scatter(yf[zero], alpha[zero], s=30, color=C["gray"],
                label=L("(a) $\\alpha_i=0$", "(a) $\\alpha_i=0$"))
axes[1].scatter(yf[free], alpha[free], s=40, color=C["green"],
                label=L("(b) $0<\\alpha_i<C$", "(b) $0<\\alpha_i<C$"))
if bnd.any():
    axes[1].scatter(yf[bnd], alpha[bnd], s=40, color=C["red"],
                    label=L("(c) $\\alpha_i=C$", "(c) $\\alpha_i=C$"))
axes[1].axvline(1.0, color="k", lw=1.0, ls=":")
axes[1].set_xlabel("$y_i f(x_i)$")
axes[1].set_ylabel(r"$\alpha_i$")
axes[1].set_title(L("支持ベクトルの三分類", "three cases of support vectors"))
axes[1].legend(fontsize=9)
plt.show()

$\alpha_i$ の大半はちょうど 0 であり、正になるのは境界のすぐそばの点だけである
（式 9.15 の和が支持ベクトルだけの和になる）。
右図の縦線 $y_if(\boldsymbol{x}_i)=1$ 上に (b) の点が乗り、
その左右に (c) と (a) が分かれる三分類が読み取れる。

## 9.4 ソフトマージンと $C$ の効果

クラスが重なるデータではスラック変数を入れる（式 9.17）。双対では箱制約
$0\le\alpha_i\le C$ が付くだけである（式 9.18）。
$C$ を $10^{-2}$ から $10^{3}$ まで動かして、マージン幅 $2/\|\boldsymbol{w}\|$、
支持ベクトル数、$\alpha_i=C$ の有界支持ベクトル数、訓練誤分類数を測る（図 9.2 に対応）。

講義ノートの演習の注意どおり、**訓練誤差は $C$ について単調とは限らない**。
単調なのは $\|\boldsymbol{w}\|$ と支持ベクトル数のほうである。

In [ ]:
Xo, yo01 = make_blobs(n_samples=60, centers=2, cluster_std=1.5, random_state=0)
Xo, yo = Xo.T, 2.0 * yo01 - 1.0          # d×n、ラベル {-1,+1}

Cs = np.logspace(-2, 3, 6)
rows = []
for Cv in Cs:
    m = SVC(kernel="linear", C=Cv, tol=1e-12).fit(Xo.T, yo)
    a = np.abs(m.dual_coef_[0])                     # |alpha_i|（支持ベクトルのみ）
    nrm = float(np.linalg.norm(m.coef_[0]))
    err = int((m.predict(Xo.T) != yo).sum())
    rows.append((Cv, nrm, 2 / nrm, m.support_.size, int((a >= Cv - 1e-6).sum()), err))

print(f"{'C':>8} {'||w||':>8} {'2/||w||':>9} {'|S|':>5} {'alpha=C':>8} {'誤分類':>6}")
for Cv, nrm, wid, ns, nb_, err in rows:
    print(f"{Cv:8.2f} {nrm:8.3f} {wid:9.3f} {ns:5d} {nb_:8d} {err:6d}")

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
axes[0].semilogx(Cs, [r[2] for r in rows], "o-", color=C["blue"])
axes[0].set_xlabel("$C$"); axes[0].set_ylabel(r"$2/\|w\|$")
axes[0].set_title(L("マージン幅", "margin width"))
axes[1].semilogx(Cs, [r[3] for r in rows], "o-", color=C["green"],
                 label=L("支持ベクトル数", "# SV"))
axes[1].semilogx(Cs, [r[4] for r in rows], "s--", color=C["red"],
                 label=L("うち $\\alpha_i=C$", "bounded SV"))
axes[1].set_xlabel("$C$"); axes[1].set_ylabel(L("個数", "count"))
axes[1].set_title(L("支持ベクトルの数", "number of support vectors"))
axes[1].legend(fontsize=9)
axes[2].semilogx(Cs, [r[5] for r in rows], "o-", color=C["orange"])
axes[2].set_xlabel("$C$"); axes[2].set_ylabel(L("誤分類数", "# misclassified"))
axes[2].set_title(L("訓練誤分類数（単調ではない）", "training errors (not monotone)"))
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, Cv in zip(axes, [0.01, 100.0]):
    m = SVC(kernel="linear", C=Cv, tol=1e-12).fit(Xo.T, yo)
    plot_linear_svm(ax, Xo, yo, m,
                    L("$C=%g$：幅 %.2f、SV %d 個" % (Cv, 2 / np.linalg.norm(m.coef_[0]), m.support_.size),
                      "$C=%g$: width %.2f, %d SVs" % (Cv, 2 / np.linalg.norm(m.coef_[0]), m.support_.size)))
axes[0].legend(fontsize=8, loc="upper left")
plt.show()

$C$ を小さくすると $\|\boldsymbol{w}\|$ が小さくなってマージンが広がり、
その帯に入り込む点（有界支持ベクトル $\alpha_i=C$）が増える。
$C$ を大きくすると帯は狭くなり支持ベクトルは減る。
上の表のとおり訓練誤分類数はこの間ほとんど動かず、単調ですらない。
ヒンジ損失の和を下げても 0-1 損失が下がるとは限らないためで、
$C$ はバイアスとバリアンスのトレードオフを制御するパラメータとして
交差確認で選ぶべきものである。

## 9.5 ヒンジ損失と他の損失関数

スラック変数を消去するとソフトマージン SVM は
「ヒンジ損失 $+$ $\ell_2$ 正則化」（式 9.22、式 9.23）に等しい。
$u=yf(\boldsymbol{x})$ の関数として 0-1 損失、ヒンジ、二乗ヒンジ、ロジスティック、指数、二乗損失を並べる
（図 9.3）。命題「ヒンジ損失は 0-1 損失の凸上界」を数値で確かめ、
$u\ge1$ で厳密に 0 になるのがヒンジと二乗ヒンジだけであることを見る。

In [ ]:
u = np.linspace(-2.0, 3.0, 601)
losses = {
    L("0-1 損失", "0-1"):            ((u <= 0).astype(float), C["gray"], "-"),
    L("ヒンジ", "hinge"):            (np.maximum(0, 1 - u), C["red"], "-"),
    L("二乗ヒンジ", "squared hinge"): (np.maximum(0, 1 - u) ** 2, C["orange"], "--"),
    L("ロジスティック", "logistic"):  (np.log2(1 + np.exp(-u)), C["blue"], "-"),
    L("指数", "exponential"):        (np.exp(-u), C["purple"], "--"),
    L("二乗", "squared"):            ((1 - u) ** 2, C["green"], ":"),
}
fig, ax = plt.subplots(figsize=(7.2, 4.6))
for name, (vals, col, ls) in losses.items():
    ax.plot(u, vals, color=col, ls=ls, lw=1.8, label=name)
ax.set_xlabel("$u = y f(x)$")
ax.set_ylabel(L("損失", "loss"))
ax.set_ylim(-0.1, 4.0)
ax.set_title(L("分類の損失関数（0-1 損失の代理損失）", "surrogate losses for 0-1 loss"))
ax.legend(fontsize=9)
plt.show()

zero_one = (u <= 0).astype(float)
for name, (vals, _, _) in losses.items():
    flat = float(np.abs(vals[u >= 1.0]).max())
    print(f"{name:>16}: 0-1 損失を下回る点の数 = {int((vals < zero_one - 1e-12).sum()):3d}"
          f",  u>=1 での最大値 = {flat:.4f}")

ヒンジ・二乗ヒンジ・ロジスティック・指数はいずれも 0-1 損失を下回らない（凸上界）。
一方、二乗損失 $(1-u)^2$ は $u$ が大きいところで増加に転じるので上界にならず、
上の出力でも 0-1 損失を下回る点が現れる。
$u\ge1$ での最大値が厳密に 0 になるのはヒンジと二乗ヒンジだけで、
この平坦部分が $\alpha_i=0$ すなわち疎性を生む。

## 9.6 カーネル SVM と $(\gamma, C)$ グリッド

双対（式 9.24）と判別関数（式 9.25）にはデータが内積の形でしか現れないので、
$\boldsymbol{x}_i^\top\boldsymbol{x}_j$ を $k(\boldsymbol{x}_i,\boldsymbol{x}_j)$ に置き換えるだけでカーネル化できる。
三日月データで線形・多項式・RBF の決定境界を比べる（図 9.4）。
続いて講義ノートのリスト 9.2 を再現し、RBF カーネルの双対を自分で解いて
`SVC` と支持ベクトル数・判別値が一致することを確かめる。

In [ ]:
Xm01, ym01 = make_moons(n_samples=200, noise=0.25, random_state=0)
Xm, ym = Xm01.T, 2.0 * ym01 - 1.0        # d×n、ラベル {-1,+1}


def plot_kernel_svm(ax, X, y, clf, title):
    x1 = np.linspace(X[0].min() - 0.5, X[0].max() + 0.5, 300)
    x2 = np.linspace(X[1].min() - 0.5, X[1].max() + 0.5, 300)
    G1, G2 = np.meshgrid(x1, x2)
    Z = clf.decision_function(np.c_[G1.ravel(), G2.ravel()]).reshape(G1.shape)
    ax.contourf(G1, G2, Z, levels=np.linspace(-2.5, 2.5, 21), cmap="RdBu_r", alpha=0.55)
    ax.contour(G1, G2, Z, levels=[-1, 0, 1], colors=[C["gray"], "k", C["gray"]],
               linestyles=["--", "-", "--"], linewidths=[1.0, 1.6, 1.0])
    ax.scatter(*X[:, y > 0], s=16, color=C["red"], edgecolors="w", linewidths=0.3)
    ax.scatter(*X[:, y < 0], s=16, color=C["blue"], edgecolors="w", linewidths=0.3)
    ax.set_title(title); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")


fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (kern, kw) in zip(axes, [("linear", {}), ("poly", {"degree": 3, "coef0": 1.0}),
                                 ("rbf", {"gamma": 1.0})]):
    m = SVC(kernel=kern, C=5.0, **kw).fit(Xm.T, ym)
    acc = float((m.predict(Xm.T) == ym).mean())
    plot_kernel_svm(ax, Xm, ym, m, f"{kern}  ({L('訓練正解率', 'train acc')} {acc:.3f}, "
                                   f"|S| = {m.support_.size})")
plt.show()

In [ ]:
def rbf_kernel(A, B, gam):
    """A: d×n_A, B: d×n_B（列がサンプル）→ 返り値は n_A×n_B。"""
    sq = (A ** 2).sum(0)[:, None] + (B ** 2).sum(0)[None, :] - 2 * A.T @ B
    return np.exp(-gam * sq)


Xk01, yk01 = make_moons(n_samples=100, noise=0.25, random_state=0)
Xk, yk = Xk01.T, 2.0 * yk01 - 1.0        # d×n
gam, Ck = 1.0, 5.0

K = rbf_kernel(Xk, Xk, gam)
a_k = svm_dual(K, yk, Ck)                                  # 9.2 節の関数をそのまま使う
sv_k = a_k > 1e-6
free_k = sv_k & (a_k < Ck - 1e-6)
b_k = float(np.mean(yk[free_k] - (a_k * yk) @ K[:, free_k]))
f_mine = (a_k * yk) @ K + b_k                              # 式 9.25 の f(x_i)

clf_k = SVC(kernel="rbf", gamma=gam, C=Ck).fit(Xk.T, yk)   # sklearn は n×d 規約
print("支持ベクトル数： 自作", int(sv_k.sum()), " / sklearn", int(clf_k.n_support_.sum()),
      f" （全 {Xk.shape[1]} 点）")
print("判別関数の最大差 =", f"{np.abs(f_mine - clf_k.decision_function(Xk.T)).max():.1e}")
print("自由な支持ベクトル (0<a<C) の数 =", int(free_k.sum()),
      " / 有界 (a=C) =", int((a_k >= Ck - 1e-6).sum()))

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

gammas = [0.1, 0.3, 1.0, 3.0, 10.0, 30.0]
Cgrid = [0.1, 1.0, 10.0, 100.0]
gs = GridSearchCV(SVC(kernel="rbf"), {"gamma": gammas, "C": Cgrid}, cv=5).fit(Xm.T, ym)
scores = gs.cv_results_["mean_test_score"].reshape(len(Cgrid), len(gammas))

fig, ax = plt.subplots(figsize=(6.4, 4.2))
im = ax.imshow(scores, cmap="viridis", aspect="auto", origin="lower")
ax.set_xticks(range(len(gammas))); ax.set_xticklabels(gammas)
ax.set_yticks(range(len(Cgrid))); ax.set_yticklabels(Cgrid)
ax.set_xlabel(r"$\gamma$"); ax.set_ylabel("$C$")
ax.set_title(L("5 分割交差確認の正解率", "5-fold CV accuracy"))
for i in range(len(Cgrid)):
    for j in range(len(gammas)):
        ax.text(j, i, f"{scores[i, j]:.3f}", ha="center", va="center",
                color="w", fontsize=8)
fig.colorbar(im, ax=ax)
ax.grid(False)
plt.show()

print("最良のパラメータ:", gs.best_params_, " CV 正解率 =", round(gs.best_score_, 4))

# gamma を大きくしすぎたときの過学習（9.8 節の警告）
print(f"\n{'gamma':>8} {'訓練正解率':>10} {'CV 正解率':>10} {'SV 割合':>8}")
for g in [0.3, 1.0, 10.0, 100.0, 1000.0]:
    m = SVC(kernel="rbf", gamma=g, C=10.0).fit(Xm.T, ym)
    cv = float(cross_val_score(SVC(kernel="rbf", gamma=g, C=10.0), Xm.T, ym, cv=5).mean())
    print(f"{g:8.1f} {float((m.predict(Xm.T) == ym).mean()):10.3f} {cv:10.3f}"
          f" {m.n_support_.sum() / Xm.shape[1]:8.2f}")

線形カーネルでは三日月データを分離できないが、多項式と RBF は曲がった境界を作る。
自作の RBF 双対と `SVC` は支持ベクトル数が一致し、判別関数の差も SLSQP の収束精度の水準に収まる。
$\gamma$ を大きくすると訓練正解率は 1 に近づく一方で交差確認の正解率は落ち、
支持ベクトル割合が 1 に近づく。9.8 節の警告「$\gamma\to\infty$ の危険」
（$\boldsymbol{K}\to\boldsymbol{I}_n$ に退化して針の山になる）が数値に現れている。

## 9.7 LOO 誤差 $\le|\mathcal{S}|/n$ の検証

定理「LOO 誤差の支持ベクトル数による評価」は $\hat R_{\mathrm{LOO}}\le|\mathcal{S}|/n$ を主張する。
$\alpha_j=0$ の点を除いても解が変わらないという議論に基づくので、これは
**ハードマージンの定理**である。有限の $C$ のソフトマージンの $k$ 分割交差確認誤差に
そのまま当てはめてはならない。

そこで線形分離可能なデータに $C=10^6$ のほぼハードマージン SVM を当て、
$\alpha_i=C$ の点が存在しない（真にハードマージン解である）ことを確認したうえで、
$n$ 回の再学習で LOO 誤差を実際に数える。

In [ ]:
def loo_vs_sv(X, y, kernel, Chard=1e6, **kw):
    """ほぼハードマージン SVM の LOO 誤差と支持ベクトル割合を測る。X は d×n。"""
    n = X.shape[1]
    full = SVC(kernel=kernel, C=Chard, tol=1e-10, **kw).fit(X.T, y)
    a = np.abs(full.dual_coef_[0])
    hard_ok = bool((a < Chard - 1e-6).all()) and int((full.predict(X.T) != y).sum()) == 0
    miss = 0
    for i in range(n):
        keep = np.ones(n, bool); keep[i] = False
        mi = SVC(kernel=kernel, C=Chard, tol=1e-10, **kw).fit(X[:, keep].T, y[keep])
        miss += int(mi.predict(X[:, [i]].T)[0] != y[i])
    return full.support_.size / n, miss / n, hard_ok, float(a.max())


Xh01, yh01 = make_blobs(n_samples=100, centers=[[-2, -2], [2, 2]], cluster_std=0.8,
                        random_state=0)
Xh, yh = Xh01.T, 2.0 * yh01 - 1.0
Xn01, yn01 = make_moons(n_samples=100, noise=0.15, random_state=0)
Xn, yn = Xn01.T, 2.0 * yn01 - 1.0

print(f"{'設定':<26}{'SV 割合':>9}{'LOO 誤差':>10}{'真にハード':>11}{'max|alpha|':>12}")
frac, loo, ok, amax = loo_vs_sv(Xh, yh, "linear")
print(f"{'blobs, 線形':<26}{frac:9.2f}{loo:10.2f}{str(ok):>11}{amax:12.1f}")
res = [("blobs, 線形", frac, loo)]
for g in [0.3, 1.0, 3.0]:
    frac, loo, ok, amax = loo_vs_sv(Xn, yn, "rbf", gamma=g)
    print(f"{'moons, RBF gamma=' + str(g):<26}{frac:9.2f}{loo:10.2f}{str(ok):>11}{amax:12.1f}")
    res.append((f"moons RBF $\\gamma$={g}", frac, loo))

fig, ax = plt.subplots(figsize=(7.0, 4.0))
idx = np.arange(len(res))
ax.bar(idx - 0.18, [r[1] for r in res], width=0.36, color=C["blue"],
       label=L("支持ベクトル割合 $|S|/n$（上界）", "SV fraction (bound)"))
ax.bar(idx + 0.18, [r[2] for r in res], width=0.36, color=C["red"],
       label=L("実測 LOO 誤差", "measured LOO error"))
ax.set_xticks(idx); ax.set_xticklabels([r[0] for r in res], fontsize=9)
ax.set_ylabel(L("割合", "fraction"))
ax.set_title(L("LOO 誤差とその上界（ほぼハードマージン、$C=10^6$）",
               "LOO error and its bound (near hard margin)"))
ax.legend(fontsize=9)
plt.show()

4 つの設定すべてで $\hat R_{\mathrm{LOO}}\le|\mathcal{S}|/n$ が成り立ち、
しかも上界はかなり緩い。`max|alpha|` が $C=10^6$ よりはるかに小さいことが、
これらが真のハードマージン解であることの確認になっている。
定理は「支持ベクトルが少なければ汎化する」という一方向だけを保証するもので、
逆は言えないことに注意する。

## 演習

**演習 9-1（双対を 1 変数に帰着する）**
$\boldsymbol{x}_1=(1,1)^\top$（$y_1=-1$）、$\boldsymbol{x}_2=(3,3)^\top$（$y_2=+1$）の 2 点に対し、
等式制約 $\sum_i\alpha_iy_i=0$ を使って双対（式 9.12）を 1 変数に帰着し、
$\alpha^\star$、$\boldsymbol{w}^\star$、$b^\star$、幾何マージンを求めよ。
主・双対の最適値が一致することも確かめよ（講義ノートの演習「2 点の SVM を手で解く」）。

**演習 9-2（$C$ と支持ベクトル数の関係）**
9.4 節のデータで、$C$ を動かしたとき単調に変化する量とそうでない量を分類せよ。
`Creg` を変えて $\|\boldsymbol{w}\|$、支持ベクトル数、訓練誤分類数の増減を調べればよい。

**演習 9-3（$\gamma$ を大きくしたときの Gram 行列）**
RBF カーネルで $\gamma\to\infty$ とすると $\boldsymbol{K}\to\boldsymbol{I}_n$ に近づくことを
$\|\boldsymbol{K}-\boldsymbol{I}_n\|_F$ で測り、そのとき双対解が $\alpha_i\to\min(1,C)$ に近づくことを確かめよ
（9.8 節の警告）。

In [ ]:
# 演習 9-1
X_ex = np.array([[1.0, 3.0],
                 [1.0, 3.0]])          # d×n = 2×2、列がサンプル
y_ex = np.array([-1.0, 1.0])
K_ex = X_ex.T @ X_ex
Q_ex = (y_ex[:, None] * y_ex[None, :]) * K_ex
print("K =", K_ex.tolist(), " Q =", Q_ex.tolist())
# TODO: 等式制約から a_1 = a_2 = a とおき、W(a) = 2a - (1/2) a^2 (Q の全成分和) を最大化して
#       a_star を求める。次に w = a_star (y_1 x_1 + y_2 x_2)、b = y_1 - w^T x_1 を計算する。
# a_star = ...
# w_ex = ...
# b_ex = ...

# 演習 9-2
# TODO: Creg を [0.01, 1, 100] と変えて SVC(kernel="linear", C=Creg) を Xo, yo に当て、
#       ||w||、support_.size、訓練誤分類数を印字して単調性を確かめる。

# 演習 9-3
# TODO: gam を [0.1, 1, 10, 100] と変えて rbf_kernel(Xk, Xk, gam) を作り、
#       np.linalg.norm(K - np.eye(n)) と svm_dual(K, yk, 1.0) の最大値を印字する。

## 演習の解答

In [ ]:
# --- 演習 9-1 ---------------------------------------------------------------
# 等式制約 -a_1 + a_2 = 0 より a_1 = a_2 = a。a^T Q a = a^2 * Q.sum() なので
# W(a) = 2a - (1/2) a^2 Q.sum()、W'(a) = 2 - a Q.sum() = 0 → a* = 2 / Q.sum()
a_star = 2.0 / Q_ex.sum()
w_ex = a_star * (y_ex * X_ex).sum(1)          # w = sum_i a_i y_i x_i
b_ex = float(y_ex[0] - w_ex @ X_ex[:, 0])     # 両点とも支持ベクトル
W_star = 2 * a_star - 0.5 * a_star ** 2 * Q_ex.sum()
print("a* =", round(a_star, 6), " w* =", np.round(w_ex, 6), " b* =", round(b_ex, 6))
print("幾何マージン 1/||w*|| =", round(1 / np.linalg.norm(w_ex), 6),
      " 2 点間距離の半分 =", round(np.linalg.norm(X_ex[:, 1] - X_ex[:, 0]) / 2, 6))
print("双対の最適値 W(a*) =", round(float(W_star), 6),
      " 主の最適値 (1/2)||w*||^2 =", round(float(0.5 * w_ex @ w_ex), 6))
chk = SVC(kernel="linear", C=1e6, tol=1e-12).fit(X_ex.T, y_ex)
print("sklearn: w =", np.round(chk.coef_[0], 6), " b =", round(float(chk.intercept_[0]), 6))

# --- 演習 9-2 ---------------------------------------------------------------
print("\n" + f"{'C':>8}{'||w||':>9}{'|S|':>6}{'誤分類':>8}")
for Cv in [0.01, 1.0, 100.0]:
    m = SVC(kernel="linear", C=Cv, tol=1e-12).fit(Xo.T, yo)
    print(f"{Cv:8.2f}{np.linalg.norm(m.coef_[0]):9.3f}{m.support_.size:6d}"
          f"{int((m.predict(Xo.T) != yo).sum()):8d}")
print("→ ||w|| は C について増加、|S| は減少。訓練誤分類数は単調ではない。")

# --- 演習 9-3 ---------------------------------------------------------------
n_k = Xk.shape[1]
print("\n" + f"{'gamma':>8}{'||K-I||_F':>12}{'max alpha':>11}{'SV 割合':>9}")
for g in [0.1, 1.0, 10.0, 100.0]:
    Kg = rbf_kernel(Xk, Xk, g)
    ag = svm_dual(Kg, yk, 1.0)
    print(f"{g:8.1f}{np.linalg.norm(Kg - np.eye(n_k)):12.3f}{ag.max():11.4f}"
          f"{float((ag > 1e-6).mean()):9.2f}")
print("→ gamma を大きくすると K は I に近づき、alpha_i は上限 min(1,C)=1 に張り付き、")
print("   すべての点が支持ベクトルになる（針の山、汎化しない）。")